# Asia GARP Research v0.3.1 — Colab
연구 전용. 실제 주문 없음. 새 ZIP을 업로드해 v0.3.1 분석을 실행합니다.
최초에는 `max_symbols=150`가 안전합니다. 공개 원천 데이터는 별도 검증이 필요합니다.


In [ ]:
%pip -q install "yfinance>=0.2.50,<2" "pandas>=2.2,<3" "numpy>=2.0,<3" "requests>=2.31,<3" "lxml>=5,<7" "pytest>=8,<10"


In [ ]:
from google.colab import files
from zipfile import ZipFile
from io import BytesIO
from pathlib import Path
import os
upload=files.upload()
assert len(upload)==1, "전체 프로젝트 ZIP 파일 1개만 업로드하세요"
target=Path("/content/garp_v031_study")
with ZipFile(BytesIO(next(iter(upload.values())))) as z: z.extractall(target)
roots=list(target.rglob("src/v031.py"))
assert len(roots)==1, "v0.3.1 프로젝트 파일 하나만 있어야 합니다"
os.chdir(roots[0].parents[1])
print("작업 경로",os.getcwd())


In [ ]:
MAX_SYMBOLS = 150  # 0=all (larger and potentially slower)
DISCOVER_MODE = "all5"  # all5 / regional5 / nikkei / off
PER_REGION = 90
FX_MODE = "live"  # live / off
EVIDENCE_MODE = "full"  # full (headlines, vendor quarter, optional DART index) / news / off
EVIDENCE_LIMIT = 8  # top provisional investigation candidates; 0..30
import subprocess
subprocess.run(["python","-m","pytest","-q","tests"],check=True)
subprocess.run(["python","-m","src.v031",
                "--max-symbols",str(MAX_SYMBOLS),"--discover-mode",DISCOVER_MODE,
                "--per-region",str(PER_REGION),"--fx-mode",FX_MODE,
                "--evidence-mode",EVIDENCE_MODE,"--evidence-limit",str(EVIDENCE_LIMIT),
                "--output","outputs"],check=True)


In [ ]:
import pandas as pd
from IPython.display import display, Markdown
from zipfile import ZipFile
out=Path("outputs")
assert (out/"03_research_report.md").exists(), "생성된 보고서가 없습니다: 실행 오류를 확인하세요"
display(Markdown((out/"03_research_report.md").read_text(encoding="utf-8")))
for filename in ["02_research_shortlist.csv","12_discovery_sources_log.csv","13_fx_normalization_audit.csv","15_cross_industry_scores.csv","18_evidence_ledger.csv"]:
    print("\n",filename)
    display(pd.read_csv(out/filename).head(20))
with ZipFile("asia_garp_v031_results.zip","w") as z:
    for f in out.iterdir():
        if f.is_file(): z.write(f,f.name)
files.download("asia_garp_v031_results.zip")
